# Home Credit：真实数据项目

本 Notebook 先读取附带的真实实验结果。主表有 307,511 行训练数据和 48,744 行测试数据；完整历史版使用 bureau、bureau_balance、previous_application、installments_payments、credit_card_balance、POS_CASH_balance 六张历史表。重新训练命令在最后，默认不执行。完整说明见 README.md 和 RESULTS_ZH.md。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
root = Path.cwd()
history_phase = "full_history"
if not (root / "main.py").exists():
    raise RuntimeError("请在 home_credit_project 文件夹中打开 notebook")

## 1. 比较模型
按 selection_cv_auc 选模型；holdout_auc 是独立检验。没有 Kaggle 测试集真值，因此没有测试集 AUC。

In [ ]:
display(pd.read_csv(root / "output/all_model_comparison.csv"))
print(json.loads((root / "output/selected_model.json").read_text()))

## 2. 看数据分布与缺失

In [ ]:
for name in ["target_distribution", "missing_values", "correlation"]:
    display(Image(filename=str(root / "output/real" / f"{name}.png")))

## 3. 主表与历史版的独立留出集表现
ROC 看排序能力，PR 看高风险客户检出质量。混淆矩阵使用开发集 OOF 选定的阈值。

In [ ]:
for name in ["roc_curve", "pr_curve", "confusion_matrix"]:
    display(Image(filename=str(root / "output" / history_phase / f"{name}.png")))

## 4. 模型解释
SHAP 横轴为树模型原始输出的贡献，不是概率百分点；红色表示特征值高。外部评分的具体来源未披露，不能推断其成因。

In [ ]:
display(Image(filename=str(root / "output" / history_phase / "shap_summary.png")))
display(pd.read_csv(root / "output" / history_phase / "shap_importance.csv").head(10))

## 5. 错误案例与最终预测
误判诊断不用于再次调整本次模型。

In [ ]:
from pathlib import Path
import pandas as pd
print("Row-level predictions are not redistributed in the public repository.")
display(pd.read_csv(root / "output" / history_phase / "error_group_summary.csv"))
print("Aggregate model selection:")
display(pd.read_csv(root / "output" / "all_model_comparison.csv"))


## 6. 可选：重新训练
将十个 CSV 按 data/README.md 命名后放入 data/，安装 requirements.txt。下面的命令需要较长时间；取消注释才执行。完整历史模型通过 --extra 训练，已使用全部收到的历史表。

In [ ]:
import subprocess, sys
# subprocess.run([sys.executable, "main.py", "--folds", "3"], check=True)
# subprocess.run([sys.executable, "enhanced.py"], check=True)
# subprocess.run([sys.executable, "enhanced.py", "--extra"], check=True)
# subprocess.run([sys.executable, "finalize_report.py"], check=True)
# subprocess.run([sys.executable, "verify_outputs.py"], check=True)